In [1]:
!pip install --upgrade pandas --quiet

In [2]:
import pandas as pd

In [3]:
import os
import requests
import zipfile
import shutil
import subprocess

In [4]:
url = "https://www.mds.gov.br/webarquivos/publicacao/sagi/microdados/02_bolsa_familia/aibf2.zip"
zip_path = "aibf2.zip"
extract_dir = "raw"
os.makedirs(extract_dir, exist_ok=True)

In [5]:
# ==============================
# 1. baixar dados
# ==============================

print("Baixando arquivo...")
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

Baixando arquivo...


In [6]:
# ==============================
# 2. extrair zip e apagá-lo
# ==============================

print("Extraindo ZIP inicial...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

os.remove(zip_path)
print("ZIP inicial removido.")

Extraindo ZIP inicial...
ZIP inicial removido.


In [7]:
# ==============================
# 3. instalar o RAR (manual + verificação)
# ==============================

def check_rar():
    try:
        subprocess.run(["unrar"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return True
    except FileNotFoundError:
        return False

if not check_rar():
    print("\n⚠️ RAR não encontrado.")
    print("Instale manualmente:")
    print("- Windows: https://www.win-rar.com/")
    print("- Linux (Debian/Ubuntu): sudo apt install unrar")
    print("- macOS (brew): brew install unrar\n")

In [8]:
# ==============================
# 4. extrair 'Microdados para analise de impacto.rar'
# ==============================

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(".rar") and "Microdados" in file:
            rar_path = os.path.join(root, file)
            print(f"Extraindo {file}...")

            subprocess.run(["unrar", "x", rar_path, root], check=False)

            os.remove(rar_path)
            print(f"{file} removido.")

Extraindo Microdados para analise descritiva_dados intermediarios.rar...

UNRAR 7.22 freeware      Copyright (c) 1993-2026 Alexander Roshal

Extracting from raw/Microdados para analise descritiva_dados intermediarios.rar

Extracting  raw/Section 16.zip                                        OK 
Extracting  raw/_Consumption Aggregates.zip                           OK 
Extracting  raw/_Misc.zip                                             OK 
Extracting  raw/01_Sintaxes.zip                                       OK 
Extracting  raw/02_Tabelas.zip                                        OK 
Extracting  raw/Section 03.zip                                        OK 
Extracting  raw/Section 04.zip                                        OK 
Extracting  raw/Section 05.zip                                        OK 
Extracting  raw/Section 06.zip                                        OK 
Extracting  raw/Section 07.zip                                        OK 
Extracting  raw/Section 08.zip        


ERROR: Bad archive raw/__MACOSX/._Microdados para analise descritiva_dados intermediarios.rar


In [9]:
# ==============================
# 5.  localizar variaveis.zip e education.zip e extrair esses zips
# ==============================

zip_targets = ["descricao das variaveis.zip", "education.zip"]

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith('.zip'):
            zip_file_path = os.path.join(root, file)
            if file in zip_targets:
                print(f"Extraindo {file}...")
    
                with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                    zip_ref.extractall(root)

            os.remove(zip_file_path)
            print(f"{file} removido.")

print("\n✅ Processo concluído!")

Section 08.zip removido.
Section 09.zip removido.
Painel_Longitudinal_2005-2009.zip removido.
prenatal care.zip removido.
Dados_Brutos_2009.zip removido.
Extraindo education.zip...
education.zip removido.
anthropometry.zip removido.
Extraindo descricao das variaveis.zip...
descricao das variaveis.zip removido.
social capital.zip removido.
_Consumption Aggregates.zip removido.
womens decision making.zip removido.
_Misc.zip removido.
01_Sintaxes.zip removido.
vaccinations.zip removido.
02_Tabelas.zip removido.
Dados_Brutos_2005.zip removido.
Section 15.zip removido.
birthweight.zip removido.
Section 14.zip removido.
labor supply.zip removido.
Section 16.zip removido.
Section 03.zip removido.
Section 07.zip removido.
Section 13.zip removido.
Section 12.zip removido.
Section 06.zip removido.
child labor.zip removido.
Section 10.zip removido.
Section 04.zip removido.
Section 05.zip removido.
Section 11.zip removido.

✅ Processo concluído!


In [10]:
# ==============================
# 6. manter apenas arquivos específicos
# ==============================

arquivos_permitidos_day_01 = [
    os.path.join("education", "education.dta"),
    os.path.join("descricao das variaveis", "descricao variaveis 1_domicilios_COMPLETE.log")
]

print("Filtrando arquivos...")

arquivos_mantidos_day01 = []

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        caminho_completo = os.path.join(root, file)
        caminho_relativo = os.path.relpath(caminho_completo, extract_dir)

        if caminho_relativo in arquivos_permitidos_day_01:
            arquivos_mantidos_day01.append(caminho_completo)
        else:
            os.remove(caminho_completo)

Filtrando arquivos...


In [11]:
# ==============================
# 7. criar pasta "day 01" e mover arquivos
# ==============================

destino = os.path.join(extract_dir, "day 01")
os.makedirs(destino, exist_ok=True)

print("Movendo arquivos selecionados...")

for arquivo in arquivos_mantidos_day01:
    nome_arquivo = os.path.basename(arquivo)
    novo_caminho = os.path.join(destino, nome_arquivo)
    shutil.move(arquivo, novo_caminho)

Movendo arquivos selecionados...


In [12]:
# ==============================
# 8. remover diretórios vazios
# ==============================

print("Removendo diretórios vazios...")

for root, dirs, files in os.walk(extract_dir, topdown=False):
    for d in dirs:
        dir_path = os.path.join(root, d)
        if not os.listdir(dir_path):
            os.rmdir(dir_path)

print("✅ Organização concluída! Arquivos estão em 'day 01'.")

Removendo diretórios vazios...
✅ Organização concluída! Arquivos estão em 'day 01'.


In [13]:
# ==============================
# 8.  Carregar o .dta SEM converter categóricas, para ver os códigos crus
# ==============================

df = pd.read_stata("raw/day 01/education.dta", convert_categoricals=False)

rotulos = pd.read_stata("raw/day 01/education.dta", iterator=True).variable_labels()

In [14]:
os.makedirs('clean/day 01/', exist_ok=True)

df_colunas = pd.DataFrame({
    "coluna": df.columns,
    "dtype": df.dtypes.astype(str),
    "rotulo": [rotulos.get(c, "") for c in df.columns],
})

df_colunas.to_csv("clean/day 01/educacao_colunas.csv", index=False, encoding="utf-8")

In [15]:
vl = pd.read_stata("raw/day 01/education.dta", iterator=True).value_labels()

with open("clean/day 01/educacao_value_labels.txt", "w", encoding="utf-8") as f:
    for nome, mapa in vl.items():
        f.write(f"== {nome} ==\n")
        for cod, sig in sorted(mapa.items()):
            f.write(f"  {cod}: {sig}\n")
        f.write("\n")

In [16]:
# (2) Amostra com todas as colunas (códigos crus)

df_amostra = df.sample(min(3000, len(df)), random_state=42)
df_amostra.to_csv("clean/day 01/educacao_amostra.csv", index=False, encoding="utf-8")

In [17]:
df.to_csv("clean/day 01/educacao_completo.csv", index=False, encoding="utf-8")

In [18]:
print("dimensões:", df.shape)

dimensões: (56367, 342)
